# Implementation of Newtons Algorithm and Steepest Descent under linear equality constraints.

In [1]:
import numpy as np
from matplotlib import pyplot as plt
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'Handin4'))
from case_studies import f1, df1, Hf1, f4, df4, Hf4, f5, df5, Hf5, x_opt


In [ ]:
def newton_equality_constrained(f, df, Hf, x0, A, b, max_iter, epsilon, rho=0.5, c1=1e-4):
    """
    Newton's Algorithm under linear equality constraints (Algorithm 10).

    Solves:
        min  f(x)
        s.t. A x + b = 0

    Stopping criterion:
        ‖(I - Aᵀ(AAᵀ)⁻¹A) ∇f(xₖ)‖ < epsilon
    i.e. the projected gradient onto the null space of A is small.
    """
    n = x0.shape[0]
    m = A.shape[0]

    xs = []
    grad_norms = []
    x_current = x0.copy()

    step_init = 1.0

    # Precompute projection matrix P = I - Aᵀ(AAᵀ)⁻¹A
    AAt = A @ A.T
    AAt_inv = np.linalg.inv(AAt)
    P = np.eye(n) - A.T @ AAt_inv @ A

    def bt_LS(x, b_step, p_k):
        """Backtracking line search with Armijo condition."""
        a = b_step
        fx = f(x)
        slope = df(x) @ p_k
        while f(x + a * p_k) > fx + c1 * a * slope:
            a = rho * a
        return a

    def compute_constrained_direction(H, g):
        """
        Compute the constrained Newton direction p_k by solving the KKT system.
        """
        try:
            np.linalg.cholesky(H)
            B = H
        except np.linalg.LinAlgError:
            eigenvalues, eigenvectors = np.linalg.eigh(H)
            B = eigenvectors @ np.diag(np.abs(eigenvalues)) @ eigenvectors.T

        zero_block = np.zeros((m, m))
        M = np.block([
            [B,          A.T        ],
            [A,          zero_block ]
        ])

        rhs = np.concatenate([-g, np.zeros(m)])
        solution = np.linalg.solve(M, rhs)
        p_k = solution[:n]
        return p_k

    b_step = step_init
    i = 0

    while i < max_iter:
        grad = df(x_current)

        # Projected gradient norm: ‖P ∇f(xₖ)‖
        projected_grad_norm = np.linalg.norm(P @ grad)

        xs.append(x_current.copy())
        grad_norms.append(np.maximum(projected_grad_norm, 1e-5 * epsilon))

        if projected_grad_norm < epsilon:
            break

        H = Hf(x_current)
        p_k = compute_constrained_direction(H, grad)

        a_k = bt_LS(x_current, b_step, p_k)
        x_current = x_current + a_k * p_k

        b_step = a_k / rho
        i += 1

    return np.array(xs), np.array(grad_norms)